In [19]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [20]:
import pandas as pd

def transform_graduate_data_by_age(file_path):
    df = pd.read_csv(file_path)

    # 1. Reshape years into rows (Melt)
    df_long = df.melt(
        id_vars=["statistics"],
        var_name="year",
        value_name="underemp_graduate"
    )

    # 2. Extract age_group and category from statistics
    def parse_stats(stat_name):
        if "_" in stat_name:
            left, right = stat_name.split("_", 1)
            if any(char.isdigit() or char in "<>≥" for char in left):
                return left.strip(), right.strip()
            return "Total", stat_name.strip()
        return "Total", stat_name.strip()

    df_long[['age_group', 'category']] = df_long['statistics'].apply(
        lambda x: pd.Series(parse_stats(x))
    )

    # 3. Split category into underemp_type and qualification
    def split_category(category):
        if pd.isna(category):
            return pd.Series([None, None], index=['underemp_type', 'qualification'])
        parts = str(category).rsplit('_', 1)
        if len(parts) == 2:
            return pd.Series([parts[0], parts[1]], index=['underemp_type', 'qualification'])
        return pd.Series([parts[0], 'total'], index=['underemp_type', 'qualification'])

    df_long[['underemp_type', 'qualification']] = df_long['category'].apply(split_category)

    # 4. Convert values to numeric and scale
    df_long['underemp_graduate'] = pd.to_numeric(df_long['underemp_graduate'], errors='coerce')
    df_long = df_long.dropna(subset=['underemp_graduate'])
    df_long['underemp_graduate'] = df_long['underemp_graduate'] * 1000

    # 5. Keep tidy structure with qualification column
    df_final = df_long[['year', 'age_group', 'underemp_type', 'qualification', 'underemp_graduate']].copy()
    df_final = df_final.sort_values(['year', 'age_group']).reset_index(drop=True)

    return df_final

In [21]:
df_final = transform_graduate_data_by_age("../../data/underemp_stats.csv")
df_final.head(10)

,year,age_group,underemp_type,qualification,underemp_graduate
0,2016,25 - 34,time,degree,6800.0
1,2016,25 - 34,time,diploma,6200.0
2,2016,25 - 34,skill,degree,132200.0
3,2016,25 - 34,skill,diploma,295600.0
4,2016,35 - 44,time,degree,3000.0
5,2016,35 - 44,time,diploma,3100.0
6,2016,35 - 44,skill,degree,35700.0
7,2016,35 - 44,skill,diploma,101200.0
8,2016,≤ 24,time,degree,4100.0
9,2016,≤ 24,time,diploma,4400.0


In [22]:
write_table(df_final, "sc_bronze", "dosm_underemployment")

Table sc_bronze.dosm_underemployment written successfully.
